# Knowledge Graph Structure

Load and visualize the built knowledge graph for THINGS-EEG2 dataset.

In [1]:
import os
import sys
from pathlib import Path

import torch
from omegaconf import OmegaConf
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.tree import Tree

console = Console()

In [2]:
def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    return start


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("PROJECT_ROOT", str(ROOT))
paths = OmegaConf.load(ROOT / "configs" / "paths" / "default.yaml")
build_kg = OmegaConf.load(ROOT / "configs" / "build_kg" / "thingseeg2.yaml")

# Merge configs to resolve interpolations
config = OmegaConf.merge({"paths": paths}, {"build_kg": build_kg})

k = config.build_kg.k
partition = config.build_kg.partition
use_open_vocab = config.build_kg.get("use_open_vocab", False)
output_format = config.build_kg.get("output_format", "pt")
save_dir = Path(str(config.build_kg.save_dir)).expanduser()

kg_path = save_dir / f"thingseeg2_kg_{partition}_k{k}.pt"

console.print(Panel.fit(str(ROOT), title="Project Root", border_style="cyan"))

config_table = Table(title="Knowledge Graph Config", show_lines=True)
config_table.add_column("Field", style="bold cyan")
config_table.add_column("Value", overflow="fold")
config_table.add_row("k (neighbors per concept)", str(k))
config_table.add_row("partition", partition)
config_table.add_row("use_open_vocab", str(use_open_vocab))
config_table.add_row("output_format", output_format)
config_table.add_row("save_dir", str(save_dir))
config_table.add_row("kg_path", str(kg_path))
config_table.add_row("exists", "yes" if kg_path.exists() else "no")
if kg_path.exists():
    config_table.add_row("size", f"{kg_path.stat().st_size:,} bytes")
console.print(config_table)

╭─ Project Root ─╮
│ D:\eegdl       │
╰────────────────╯

                                  Knowledge Graph Config                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field                     ┃ Value                                                       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ k (neighbors per concept) │ 5                                                           │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ partition                 │ training                                                    │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ use_open_vocab            │ True                                                        │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ output_format             │ all                                                         │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ save_dir                  │ D:\eegdl\data\knowledge_graphs                              │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ kg_path                   │ D:\eegdl\data\knowledge_graphs\thingseeg2_kg_training_k5.pt │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ exists                    │ yes                                                         │
├───────────────────────────┼─────────────────────────────────────────────────────────────┤
│ size                      │ 3,765,123 bytes                                             │
└───────────────────────────┴─────────────────────────────────────────────────────────────┘

In [3]:
def tensor_summary(value):
    summary = {
        "shape": tuple(value.shape),
        "dtype": str(value.dtype),
        "numel": f"{value.numel():,}",
    }
    if value.numel() > 0 and not value.dtype.is_floating_point:
        summary.update({"min": value.min().item(), "max": value.max().item()})
    return summary


def add_value_to_tree(tree, name, value, max_items=5):
    if torch.is_tensor(value):
        branch = tree.add(f"[bold green]{name}[/]: Tensor")
        for key, item in tensor_summary(value).items():
            branch.add(f"[cyan]{key}[/]: {item}")
    elif isinstance(value, dict):
        branch = tree.add(f"[bold blue]{name}[/]: dict (keys={len(value)})")
        for i, (key, item) in enumerate(list(value.items())[:max_items]):
            add_value_to_tree(branch, str(key), item, max_items=max_items)
        if len(value) > max_items:
            branch.add(f"... {len(value) - max_items} more")
    elif isinstance(value, (list, tuple)):
        branch = tree.add(f"[bold magenta]{name}[/]: {type(value).__name__} (len={len(value)})")
        for idx, item in enumerate(value[:max_items]):
            add_value_to_tree(branch, f"[{idx}]", item, max_items=max_items)
        if len(value) > max_items:
            branch.add(f"... {len(value) - max_items} more")
    else:
        tree.add(f"[bold]{name}[/]: {type(value).__name__} = {value!r}")

In [4]:
console.rule("[bold cyan]Loading Knowledge Graphs")

# Load training KG
kg_path_train = save_dir / f"thingseeg2_kg_training_k{k}.pt"
kg_data_train = torch.load(kg_path_train, map_location="cpu", weights_only=False) if kg_path_train.exists() else None

# Load test KG
kg_path_test = save_dir / f"thingseeg2_kg_test_k{k}.pt"
kg_data_test = torch.load(kg_path_test, map_location="cpu", weights_only=False) if kg_path_test.exists() else None

console.print(f"[green]✓[/] Training KG loaded: {kg_path_train.exists()}")
console.print(f"[green]✓[/] Test KG loaded: {kg_path_test.exists()}")

# Use training as default for structure display
kg_data = kg_data_train if kg_data_train else kg_data_test

tree = Tree("[bold]knowledge_graph (training)[/]")
add_value_to_tree(tree, "root", kg_data, max_items=4)
console.print(tree)

──────────────────────────────────────────── Loading Knowledge Graphs ─────────────────────────────────────────────

✓ Training KG loaded: True

✓ Test KG loaded: True

knowledge_graph (training)
└── root: dict (keys=4)
    ├── concepts: dict (keys=1753)
    │   ├── 0: dict (keys=3)
    │   │   ├── name: str = 'aardvark'
    │   │   ├── source: str = 'thingseeg2'
    │   │   └── emb_type: str = 'image_avg'
    │   ├── 1: dict (keys=3)
    │   │   ├── name: str = 'abacus'
    │   │   ├── source: str = 'thingseeg2'
    │   │   └── emb_type: str = 'image_avg'
    │   ├── 2: dict (keys=3)
    │   │   ├── name: str = 'accordion'
    │   │   ├── source: str = 'thingseeg2'
    │   │   └── emb_type: str = 'image_avg'
    │   ├── 3: dict (keys=3)
    │   │   ├── name: str = 'acorn'
    │   │   ├── source: str = 'thingseeg2'
    │   │   └── emb_type: str = 'image_avg'
    │   └── ... 1749 more
    ├── concept_neighbors: Tensor
    │   ├── shape: (1753, 5)
    │   ├── dtype: torch.int64
    │   ├── numel: 8,765
    │   ├── min: 1654
    │   └── max: 1752
    ├── concept_neighbor_scores: Tensor
    │   ├── shape: (1753, 5)
    │   ├── dtype: torch.float32
    │   └── numel: 8,765
    └── concept_embeddings: Tensor
        ├── shape: (1753, 512)
        ├── dtype: torch.float32
        └── numel: 897,536

In [5]:
# Configuration for concept display
train_display_concept_ids = [0, 1, 2, 3, 4]  # Which concepts to display
test_display_concept_ids = [5, 6, 7, 8, 9]  # Which concepts to display in test set
num_neighbors_to_show = 5  # How many neighbors per concept

In [7]:
def display_concept_neighbors(kg_data, partition_name, concept_ids, num_neighbors):
    """Display concept neighbors for specified concept IDs"""
    if kg_data is None:
        console.print(f"[yellow]⚠[/] {partition_name} KG not available")
        return
    
    if "concepts" in kg_data:
        # Open vocabulary mode
        table = Table(
            title=f"{partition_name} - Concept Neighbors ({len(concept_ids)} concepts, {num_neighbors} neighbors each)",
            show_lines=True
        )
        table.add_column("Concept ID", style="bold cyan")
        table.add_column("Name", style="white")
        table.add_column("Neighbor Concepts", overflow="fold")

        for concept_id in concept_ids:
            if concept_id >= kg_data["concept_neighbors"].shape[0]:
                continue
            name = kg_data["concepts"][concept_id]["name"]
            neighbor_ids = kg_data["concept_neighbors"][concept_id][:num_neighbors].tolist()
            neighbor_names = [
                kg_data["concepts"][nid]["name"] for nid in neighbor_ids if nid in kg_data["concepts"]
            ]
            table.add_row(str(concept_id), name, str(neighbor_names))

        console.print(table)
    elif "concept_to_images" in kg_data:
        # Dataset-only mode
        table = Table(
            title=f"{partition_name} - Concept Neighbors ({len(concept_ids)} concepts, {num_neighbors} neighbors each)",
            show_lines=True
        )
        table.add_column("Concept ID", style="bold cyan")
        table.add_column("# Images", justify="right")
        table.add_column("Neighbor Concepts", overflow="fold")

        for concept_id in concept_ids:
            if concept_id >= kg_data["concept_neighbors"].shape[0]:
                continue
            n_images = len(kg_data["concept_to_images"][concept_id])
            neighbors = kg_data["concept_neighbors"][concept_id][:num_neighbors].tolist()
            table.add_row(str(concept_id), str(n_images), str(neighbors))

        console.print(table)

# Display training set
display_concept_neighbors(kg_data_train, "Training", train_display_concept_ids, num_neighbors_to_show)

# Display test set
display_concept_neighbors(kg_data_test, "Test", test_display_concept_ids, num_neighbors_to_show)

                            Training - Concept Neighbors (5 concepts, 5 neighbors each)                            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Concept ID ┃ Name            ┃ Neighbor Concepts                                                                ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0          │ aardvark        │ ['four-legged animal', 'mammal', 'small animal', 'large animal', 'animal']       │
├────────────┼─────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 1          │ abacus          │ ['percussion instrument', 'musical instrument', 'keyboard instrument', 'ball     │
│            │                 │ sports equipment', 'string instrument']                                          │
├────────────┼─────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 2          │ accordion       │ ['keyboard instrument', 'musical instrument', 'percussion instrument', 'string   │
│            │                 │ instrument', 'wind instrument']                                                  │
├────────────┼─────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 3          │ acorn           │ ['fruit', 'edible object', 'ball-like object', 'plant', 'vegetable']             │
├────────────┼─────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 4          │ air conditioner │ ['electronic device', 'hard object', 'shiny object', 'portable object',          │
│            │                 │ 'box-like object']                                                               │
└────────────┴─────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

                              Test - Concept Neighbors (5 concepts, 5 neighbors each)                              
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Concept ID ┃ Name         ┃ Neighbor Concepts                                                                   ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 5          │ baseball bat │ ['ball sports equipment', 'wind instrument', 'sports equipment', 'racket sports     │
│            │              │ equipment', 'cutting tool']                                                         │
├────────────┼──────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ 6          │ basil        │ ['leafy plant', 'green plant', 'plant', 'vegetable', 'ball-like object']            │
├────────────┼──────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ 7          │ basketball   │ ['ball sports equipment', 'sports equipment', 'ball-like object', 'spherical        │
│            │              │ object', 'wooden object']                                                           │
├────────────┼──────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ 8          │ bassoon      │ ['wind instrument', 'musical instrument', 'percussion instrument', 'string          │
│            │              │ instrument', 'tube-like object']                                                    │
├────────────┼──────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ 9          │ baton4       │ ['elongated object', 'hard object', 'sports equipment', 'rod-like object',          │
│            │              │ 'tube-like object']                                                                 │
└────────────┴──────────────┴─────────────────────────────────────────────────────────────────────────────────────┘